In [1]:
from pathlib import Path

from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

ea = {"euler_angles": (30, -30, 0)}

In [2]:
box = Box(p1=(0, 0, 0), p2=(10, 10, 20))
box.mat("box")
box.faces.name = "box"
box.faces.Min(X).name = "left"
box.faces.Max(X).name = "right"
box.faces.Min(Y).name = "bottom"
box.faces.Max(Y).name = "top"
box.faces.Min(Z).name = "rear"
box.faces.Max(Z).name = "front"
box.edges.name = "edges"

cyl = Cylinder(p=(5, 5, 0), d=Z, r=2, h=20)
cyl.mat("tube")
cyl.faces.name = "interface"
cyl.faces.Min(Z).name = "inlet"
cyl.faces.Max(Z).name = "outlet"

shape = box - cyl

ngmesh = OCCGeometry(shape).GenerateMesh(maxh=1)

mesh = Mesh(ngmesh)
mesh.Curve(3)
Draw(mesh, **ea)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

BaseWebGuiScene

In [3]:
# Material Parameters
E = 21e3
nu = 0.35
mu = E / 2 / (1 + nu)
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

# Initialize FE Space
fes = VectorH1(mesh, order=2, dirichlet="bottom")

# Initialize symbolic u as trial function
# for constructing symbolic term (such as NeoHooke)
u = fes.TrialFunction()

# Components for Strain Energy
I = Id(mesh.dim)
F = I + Grad(u)
C = F.trans * F
E = 0.5 * (C - I)


def Pow(a, b):
    return a**b  # exp (log(a)*b)


def NeoHooke(C):
    return 0.5 * mu * (Trace(C - I) + 2 * mu / lam * Pow(Det(C), -lam / 2 / mu) - 1)


# Constant Loading
# force = CoefficientFunction((0, 10, 0))

# Gravity Loading
# rho = 1100e-6
# g = 9.81e+3
# fgrav = -(rho * g)
# force = CoefficientFunction((0, fgrav, 0))

# Normal Loading
# fnormal = 10
n = specialcf.normal(mesh.dim)

# Propagating Wave Loading
time = Parameter(0)
v_pulse = 8.0  # Velocity of the pulse
width = 2.0  # How "long" the pulse is
peak_p = 10  # Maximum pressure
z_center = v_pulse * time  # The moving center of the pulse
# The Pressure CoefficientFunction
# This creates a 'hump' of pressure that moves along the Z axis
pulse_pressure = peak_p * exp(-((z - z_center) ** 2) / (2 * width**2))

# Stiffness Matrices
a = BilinearForm(fes, symmetric=False)
a += Variation(NeoHooke(C).Compile() * dx)
a += Variation((pulse_pressure * InnerProduct(n, u)).Compile() * ds("interface"))

# Initialize solution u in a format of grid function
u = GridFunction(fes)

# Allocate the memory for calculated components in NR
res = u.vec.CreateVector()  # residual
w = u.vec.CreateVector()  # incremental displacement solution (from NR)

In [4]:
# Define solving parameters
n_timestep = 50
n_nr = 5
dt = 0.1

# Solve with time stepping
for t_step in range(n_timestep):
    current_t = t_step * dt
    time.Set(current_t)
    print(f"Time Step: {t_step + 1}/{n_timestep}")
    print(f"Time: {current_t:.2f}s")

    for i_nr in range(n_nr):
        print(f"\tNewton iteration: {i_nr + 1}/{n_nr}")
        print(f"\t\tEnergy: {a.Energy(u.vec):.6f}")
        a.Apply(u.vec, res)
        a.AssembleLinearization(u.vec)
        inv = a.mat.Inverse(fes.FreeDofs())
        w.data = inv * res
        print(f"\t\tErr^2: {InnerProduct(w, res):.6e}")
        u.vec.data -= w

    Draw(u, mesh, deformation=True, scale=1e3, **ea)

Time Step: 1/50
Time: 0.00s
	Newton iteration: 1/5
		Energy: -971478.779911
		Err^2: 3.213707e-01
	Newton iteration: 2/5
		Energy: -971478.940612
		Err^2: 3.098018e-08
	Newton iteration: 3/5
		Energy: -971478.940612
		Err^2: 5.478074e-21
	Newton iteration: 4/5
		Energy: -971478.940612
		Err^2: 4.066591e-25
	Newton iteration: 5/5
		Energy: -971478.940612
		Err^2: 3.699312e-25


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

Time Step: 2/50
Time: 0.10s
	Newton iteration: 1/5
		Energy: -971479.002621
		Err^2: 2.187151e-02
	Newton iteration: 2/5
		Energy: -971479.013557
		Err^2: 6.097109e-11
	Newton iteration: 3/5
		Energy: -971479.013557
		Err^2: 7.963041e-25
	Newton iteration: 4/5
		Energy: -971479.013557
		Err^2: 3.806656e-25
	Newton iteration: 5/5
		Energy: -971479.013557
		Err^2: 3.881932e-25


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

Time Step: 3/50
Time: 0.20s
	Newton iteration: 1/5
		Energy: -971479.054670
		Err^2: 2.135989e-02
	Newton iteration: 2/5
		Energy: -971479.065350
		Err^2: 1.156918e-10
	Newton iteration: 3/5
		Energy: -971479.065350
		Err^2: 8.501937e-25
	Newton iteration: 4/5
		Energy: -971479.065350
		Err^2: 3.842797e-25
	Newton iteration: 5/5
		Energy: -971479.065350
		Err^2: 3.825190e-25


...

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

Time Step: 49/50
Time: 4.80s
	Newton iteration: 1/5
		Energy: -971478.779911
		Err^2: 6.432963e-26
	Newton iteration: 2/5
		Energy: -971478.779911
		Err^2: 6.124935e-26
	Newton iteration: 3/5
		Energy: -971478.779911
		Err^2: 6.507793e-26
	Newton iteration: 4/5
		Energy: -971478.779911
		Err^2: 6.400413e-26
	Newton iteration: 5/5
		Energy: -971478.779911
		Err^2: 6.268597e-26


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…

Time Step: 50/50
Time: 4.90s
	Newton iteration: 1/5
		Energy: -971478.779911
		Err^2: 6.143408e-26
	Newton iteration: 2/5
		Energy: -971478.779911
		Err^2: 6.504826e-26
	Newton iteration: 3/5
		Energy: -971478.779911
		Err^2: 6.389047e-26
	Newton iteration: 4/5
		Energy: -971478.779911
		Err^2: 6.212178e-26
	Newton iteration: 5/5
		Energy: -971478.779911
		Err^2: 6.507059e-26


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (3…